# 📊 Análisis Univariado y Bivariado
**Telco Customer Churn | ITY1101 — Evaluación Parcial N°3**

Este notebook incluye:
- Estadísticas descriptivas (media, mediana, moda, percentiles)
- Análisis univariado (distribución de cada variable)
- Análisis bivariado (relación entre variables y Churn)
- Matriz de correlación

## 📦 1. Importaciones y configuración

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configurar ruta base del proyecto
os.chdir('/Users/nicolasherrera/Telco_Churn')

print('✅ Librerías importadas correctamente')

## 📥 2. Cargar Dataset

In [ ]:
df = pd.read_csv('IA_Proyecto/data/telco_limpio.csv')
print(f'✅ Dataset cargado: {df.shape[0]} filas | {df.shape[1]} columnas')
print(f'\nPrimeras 5 filas:')
df.head()

## 📊 3. Estadísticas Descriptivas (Medidas de Calidad)

In [ ]:
print('=' * 60)
print('ESTADÍSTICAS DESCRIPTIVAS — Variables Numéricas')
print('=' * 60)
desc = df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe(percentiles=[.25, .5, .75])
print(desc.round(2))

print('\n' + '=' * 60)
print('VALORES NULOS POR COLUMNA')
print('=' * 60)
nulos = df.isnull().sum()
print(f'Total nulos: {nulos.sum()} (pipeline DataOps los trató correctamente ✅)')

print('\n' + '=' * 60)
print('DISTRIBUCIÓN VARIABLE OBJETIVO — Churn')
print('=' * 60)
churn_dist = df['Churn'].value_counts()
for val, count in churn_dist.items():
    print(f'  {val}: {count} clientes ({count/len(df)*100:.1f}%)')

## 📈 4. Análisis Univariado — Variables Numéricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
colores  = ['#3498db', '#9b59b6', '#e67e22']

for ax, col, color in zip(axes, cols_num, colores):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(),   color='red',   linestyle='--', label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(f'Distribución de {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Análisis Univariado — Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('IA_Proyecto/data/univariado_numerico.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Análisis univariado numérico generado')

## 📊 5. Análisis Univariado — Variables Categóricas

In [ ]:
cols_cat = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, cols_cat):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color='#3498db', alpha=0.85, edgecolor='white')
    ax.set_title(f'Distribución de {col}', fontweight='bold')
    ax.set_ylabel('Cantidad')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val}', ha='center', fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.suptitle('Análisis Univariado — Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('IA_Proyecto/data/univariado_categorico.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Análisis univariado categórico generado')

## 🔗 6. Análisis Bivariado — Variables vs Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']

for ax, col in zip(axes, cols_num):
    churn_yes = df[df['Churn'] == 'Yes'][col]
    churn_no  = df[df['Churn'] == 'No'][col]
    ax.hist(churn_no,  bins=25, alpha=0.6, color='#2ecc71', label='No Churn')
    ax.hist(churn_yes, bins=25, alpha=0.6, color='#e74c3c', label='Churn')
    ax.set_title(f'{col} vs Churn', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Análisis Bivariado — Variables Numéricas vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('IA_Proyecto/data/bivariado_numerico.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Análisis bivariado numérico generado')

In [ ]:
# Tasa de Churn por variables categóricas
cols_cat = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, cols_cat):
    tasa = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).sort_values(ascending=False)
    bars = ax.bar(tasa.index, tasa.values, color='#e74c3c', alpha=0.85, edgecolor='white')
    ax.set_title(f'Tasa de Churn por {col}', fontweight='bold')
    ax.set_ylabel('Tasa de Churn (%)')
    for bar, val in zip(bars, tasa.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.suptitle('Análisis Bivariado — Tasa de Churn por Variable Categórica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('IA_Proyecto/data/bivariado_categorico.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Análisis bivariado categórico generado')

## 🔥 7. Matriz de Correlación

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_num = df.copy()
le = LabelEncoder()
for col in df_num.select_dtypes(include=['object']).columns:
    df_num[col] = le.fit_transform(df_num[col])

corr = df_num.corr()

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        val = corr.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color='black' if abs(val) < 0.7 else 'white')

ax.set_title('Matriz de Correlación — Telco Customer Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('IA_Proyecto/data/matriz_correlacion.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Matriz de correlación generada')

## 📋 8. Hallazgos Principales

In [ ]:
print('=' * 60)
print('  HALLAZGOS PRINCIPALES DEL ANÁLISIS')
print('=' * 60)

# Tenure promedio por Churn
tenure_churn = df.groupby('Churn')['tenure'].mean()
print(f'\n📌 Tenure promedio:')
print(f'   Clientes que se van   : {tenure_churn["Yes"]:.1f} meses')
print(f'   Clientes que se quedan: {tenure_churn["No"]:.1f} meses')
print(f'   → Clientes nuevos tienen más riesgo de Churn')

# MonthlyCharges promedio por Churn
mc_churn = df.groupby('Churn')['MonthlyCharges'].mean()
print(f'\n📌 Cargo mensual promedio:')
print(f'   Clientes que se van   : ${mc_churn["Yes"]:.2f}')
print(f'   Clientes que se quedan: ${mc_churn["No"]:.2f}')
print(f'   → Clientes con cobros más altos tienen más riesgo de Churn')

# Contrato
contrato_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x=='Yes').sum()/len(x)*100
).sort_values(ascending=False)
print(f'\n📌 Tasa de Churn por contrato:')
for contrato, tasa in contrato_churn.items():
    print(f'   {contrato}: {tasa:.1f}%')
print(f'   → Contratos mes a mes tienen mayor riesgo de Churn')

print('=' * 60)